In [15]:
import pandas as pd

In [32]:
weather_data=pd.read_csv("data_clean.csv")

In [33]:
weather_data

,Unnamed: 0,Ozone,Solar.R,Wind,Temp C,Month,Day,Year,Temp,Weather
0,1,41.0,190.0,7.4,67,5,1,2010,67,S
1,2,36.0,118.0,8.0,72,5,2,2010,72,C
2,3,12.0,149.0,12.6,74,5,3,2010,74,PS
3,4,18.0,313.0,11.5,62,5,4,2010,62,S
4,5,NaN,NaN,14.3,56,5,5,2010,56,S
...,...,...,...,...,...,...,...,...,...,...
153,154,41.0,190.0,7.4,67,5,1,2010,67,C
154,155,30.0,193.0,6.9,70,9,26,2010,70,PS
155,156,NaN,145.0,13.2,77,9,27,2010,77,S
156,157,14.0,191.0,14.3,75,9,28,2010,75,S


In [18]:
weather_data.isnull().sum()

Unnamed: 0     0
Ozone         38
Solar.R        7
Wind           0
Temp C         0
Month          0
Day            0
Year           0
Temp           0
Weather        3
dtype: int64

In [19]:
weather_data.dtypes

Unnamed: 0      int64
Ozone         float64
Solar.R       float64
Wind          float64
Temp C         object
Month          object
Day             int64
Year            int64
Temp            int64
Weather        object
dtype: object

In [22]:
weather_data.describe(include="all")

,Ozone,Solar.R,Wind,Month,Day,Year,Temp,Weather
count,120.000000,151.000000,158.000000,158,158.000000,158.0,158.000000,155
unique,NaN,NaN,NaN,6,NaN,NaN,NaN,3
top,NaN,NaN,NaN,9,NaN,NaN,NaN,S
freq,NaN,NaN,NaN,34,NaN,NaN,NaN,59
mean,41.583333,185.403974,9.957595,NaN,16.006329,2010.0,77.727848,NaN
std,32.620709,88.723103,3.511261,NaN,8.997166,0.0,9.377877,NaN
min,1.000000,7.000000,1.700000,NaN,1.000000,2010.0,56.000000,NaN
25%,18.000000,119.000000,7.400000,NaN,8.000000,2010.0,72.000000,NaN
50%,30.500000,197.000000,9.700000,NaN,16.000000,2010.0,78.500000,NaN
75%,61.500000,257.000000,11.875000,NaN,24.000000,2010.0,84.000000,NaN


In [35]:
weather_data["Weather"].unique() #to check whats the unique entries in output column

array(['S', 'C', 'PS', nan], dtype=object)

In [26]:
weather_data["Weather"].value_counts(dropna=False) #dropna=false to know how many null entries, by default it is true

Weather
S      59
C      49
PS     47
NaN     3
Name: count, dtype: int64

## Data Preprocessing 

### Stage 1 - Data cleaning - Removing null & duplicates; Mean,median,Mode imputation if possible; Changing datatypes

In [34]:
weather_data.drop(["Unnamed: 0","Temp C"],axis=1,inplace=True) #del can be used for single column; here we use drop 
weather_data #Unnamed C doesnt contribute to predictions; Temp C is duplicate column

,Ozone,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,41.0,190.0,7.4,5,1,2010,67,S
1,36.0,118.0,8.0,5,2,2010,72,C
2,12.0,149.0,12.6,5,3,2010,74,PS
3,18.0,313.0,11.5,5,4,2010,62,S
4,NaN,NaN,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...,...
153,41.0,190.0,7.4,5,1,2010,67,C
154,30.0,193.0,6.9,9,26,2010,70,PS
155,NaN,145.0,13.2,9,27,2010,77,S
156,14.0,191.0,14.3,9,28,2010,75,S


In [36]:
weather_data.isnull().sum()

Ozone      38
Solar.R     7
Wind        0
Month       0
Day         0
Year        0
Temp        0
Weather     3
dtype: int64

In [44]:
38/158 #(For Ozone)almost 25% data is not available for OZone parameter; if we use MMM imputation it would lead to misinfo so we remove that column

0.24050632911392406

In [41]:
del weather_data["Ozone"]

In [43]:
weather_data #Ozone is deleted here as 25% is null

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.0,7.4,5,1,2010,67,S
1,118.0,8.0,5,2,2010,72,C
2,149.0,12.6,5,3,2010,74,PS
3,313.0,11.5,5,4,2010,62,S
4,NaN,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.0,7.4,5,1,2010,67,C
154,193.0,6.9,9,26,2010,70,PS
155,145.0,13.2,9,27,2010,77,S
156,191.0,14.3,9,28,2010,75,S


In [46]:
7/158 #(For SolarR) this NAN entries shows negligible % so we can make some imputations

0.04430379746835443

In [49]:
weather_data[weather_data["Solar.R"].isna()]

,Solar.R,Wind,Month,Day,Year,Temp,Weather
4,NaN,14.3,5,5,2010,56,S
5,NaN,14.9,5,6,2010,66,C
10,NaN,6.9,5,11,2010,74,C
26,NaN,8.0,5,27,2010,57,PS
95,NaN,6.9,8,4,2010,86,NaN
96,NaN,7.4,8,5,2010,85,S
97,NaN,4.6,8,6,2010,87,C


In [66]:
weather_data.groupby("Weather")["Solar.R"].mean().round() #Finding mean 

Weather
C     192.0
PS    174.0
S     193.0
Name: Solar.R, dtype: float64

In [ ]:
#as per the data when the weather is sunny, the avg is 193; PS-174; C-192

In [67]:
weather_data["Solar.R"]=weather_data.groupby("Weather")["Solar.R"].transform(lambda x:x.fillna(x.mean().round(2)))
weather_data #using lambda we fill the nan with mean values in Solar.R column

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,S
1,118.00,8.0,5,2,2010,72,C
2,149.00,12.6,5,3,2010,74,PS
3,313.00,11.5,5,4,2010,62,S
4,193.33,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,C
154,193.00,6.9,9,26,2010,70,PS
155,145.00,13.2,9,27,2010,77,S
156,191.00,14.3,9,28,2010,75,S


In [62]:
weather_data.isna().sum() #null values is now reduced from 7 to 3 

Solar.R    3
Wind       0
Month      0
Day        0
Year       0
Temp       0
Weather    3
dtype: int64

In [63]:
weather_data[weather_data["Solar.R"].isna()] #wherever the o/p is nan, I/P is also nan ; so we drop the output column

,Solar.R,Wind,Month,Day,Year,Temp,Weather
87,NaN,12.0,7,27,2010,86,NaN
93,NaN,13.8,8,2,2010,81,NaN
95,NaN,6.9,8,4,2010,86,NaN


In [64]:
weather_data.dropna( inplace=True)

In [65]:
weather_data

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,S
1,118.00,8.0,5,2,2010,72,C
2,149.00,12.6,5,3,2010,74,PS
3,313.00,11.5,5,4,2010,62,S
4,193.33,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,C
154,193.00,6.9,9,26,2010,70,PS
155,145.00,13.2,9,27,2010,77,S
156,191.00,14.3,9,28,2010,75,S


In [ ]:
##Changing datatypes

In [69]:
weather_data.dtypes 

Solar.R    float64
Wind       float64
Month       object
Day          int64
Year         int64
Temp         int64
Weather     object
dtype: object

In [72]:
weather_data["Month"].unique() #By mistake there is alphabet entry, thats y it shows object datatype; so we have to replace that

array(['5', 'May', '6', '7', '8', '9'], dtype=object)

In [74]:
weather_data.head(30) #entry no 23 has "May"

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,S
1,118.00,8.0,5,2,2010,72,C
2,149.00,12.6,5,3,2010,74,PS
3,313.00,11.5,5,4,2010,62,S
4,193.33,14.3,5,5,2010,56,S
5,192.39,14.9,5,6,2010,66,C
6,299.00,8.6,5,7,2010,65,PS
7,99.00,13.8,5,8,2010,59,C
8,19.00,20.1,5,9,2010,61,PS
9,194.00,8.6,5,10,2010,69,S


In [75]:
weather_data["Month"].replace(to_replace = "May",value=5,inplace=True)

C:\Users\Mohana Priya\AppData\Local\Temp\ipykernel_11216\3637294146.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  weather_data["Month"].replace(to_replace = "May",value=5,inplace=True)


In [77]:
weather_data.head(30)

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,S
1,118.00,8.0,5,2,2010,72,C
2,149.00,12.6,5,3,2010,74,PS
3,313.00,11.5,5,4,2010,62,S
4,193.33,14.3,5,5,2010,56,S
5,192.39,14.9,5,6,2010,66,C
6,299.00,8.6,5,7,2010,65,PS
7,99.00,13.8,5,8,2010,59,C
8,19.00,20.1,5,9,2010,61,PS
9,194.00,8.6,5,10,2010,69,S


In [80]:
weather_data.dtypes #even aft changing to numeric the dtypes for "month" is still object so we have to change type

Solar.R    float64
Wind       float64
Month       object
Day          int64
Year         int64
Temp         int64
Weather     object
dtype: object

In [81]:
weather_data["Month"]=weather_data["Month"].astype("int")
weather_data

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,S
1,118.00,8.0,5,2,2010,72,C
2,149.00,12.6,5,3,2010,74,PS
3,313.00,11.5,5,4,2010,62,S
4,193.33,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,C
154,193.00,6.9,9,26,2010,70,PS
155,145.00,13.2,9,27,2010,77,S
156,191.00,14.3,9,28,2010,75,S


In [83]:
weather_data.dtypes #now the datatype is changed

Solar.R    float64
Wind       float64
Month        int64
Day          int64
Year         int64
Temp         int64
Weather     object
dtype: object

In [ ]:
## All the Input features dtypes are now changed to numeric

## Stage 2-Data Transform 

In [ ]:
# first will make the copy

In [84]:
weather_data_copy=weather_data.copy()

In [85]:
weather_data_copy

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,S
1,118.00,8.0,5,2,2010,72,C
2,149.00,12.6,5,3,2010,74,PS
3,313.00,11.5,5,4,2010,62,S
4,193.33,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,C
154,193.00,6.9,9,26,2010,70,PS
155,145.00,13.2,9,27,2010,77,S
156,191.00,14.3,9,28,2010,75,S


In [ ]:
### Label encoder

In [87]:
from sklearn.preprocessing import LabelEncoder

In [92]:
label_encode=LabelEncoder()
weather_data_copy["Weather"]=label_encode.fit_transform(weather_data_copy["Weather"]) 
weather_data_copy #encoding is done 

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,2
1,118.00,8.0,5,2,2010,72,0
2,149.00,12.6,5,3,2010,74,1
3,313.00,11.5,5,4,2010,62,2
4,193.33,14.3,5,5,2010,56,2
...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,0
154,193.00,6.9,9,26,2010,70,1
155,145.00,13.2,9,27,2010,77,2
156,191.00,14.3,9,28,2010,75,2


In [94]:
weather_data_copy.dtypes #now the dtype os weather is changed from obj to int

Solar.R    float64
Wind       float64
Month        int64
Day          int64
Year         int64
Temp         int64
Weather      int64
dtype: object

In [ ]:
### OneHotencoder -using Pandas

In [108]:
weather_data_copy2=weather_data.copy()
weather_data_copy2

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,S
1,118.00,8.0,5,2,2010,72,C
2,149.00,12.6,5,3,2010,74,PS
3,313.00,11.5,5,4,2010,62,S
4,193.33,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,C
154,193.00,6.9,9,26,2010,70,PS
155,145.00,13.2,9,27,2010,77,S
156,191.00,14.3,9,28,2010,75,S


In [109]:
weather_data_copy2=pd.get_dummies(data=weather_data_copy2,columns=["Weather"],dtype="int") #if we give dtype it values will change from boolean to int

In [110]:
weather_data_copy2

,Solar.R,Wind,Month,Day,Year,Temp,Weather_C,Weather_PS,Weather_S
0,190.00,7.4,5,1,2010,67,0,0,1
1,118.00,8.0,5,2,2010,72,1,0,0
2,149.00,12.6,5,3,2010,74,0,1,0
3,313.00,11.5,5,4,2010,62,0,0,1
4,193.33,14.3,5,5,2010,56,0,0,1
...,...,...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,1,0,0
154,193.00,6.9,9,26,2010,70,0,1,0
155,145.00,13.2,9,27,2010,77,0,0,1
156,191.00,14.3,9,28,2010,75,0,0,1


In [112]:
weather_data_copy2.dtypes #now op column is changed to int

Solar.R       float64
Wind          float64
Month           int64
Day             int64
Year            int64
Temp            int64
Weather_C       int64
Weather_PS      int64
Weather_S       int64
dtype: object

In [ ]:
### OneHotencoder -using sklearn

In [113]:
from sklearn.preprocessing import OneHotEncoder

In [159]:
weather_data_copy3=weather_data.copy()
weather_data_copy3

,Solar.R,Wind,Month,Day,Year,Temp,Weather
0,190.00,7.4,5,1,2010,67,S
1,118.00,8.0,5,2,2010,72,C
2,149.00,12.6,5,3,2010,74,PS
3,313.00,11.5,5,4,2010,62,S
4,193.33,14.3,5,5,2010,56,S
...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,C
154,193.00,6.9,9,26,2010,70,PS
155,145.00,13.2,9,27,2010,77,S
156,191.00,14.3,9,28,2010,75,S


In [160]:
one_hot_encode=OneHotEncoder(sparse_output=False)

In [161]:
encoded=one_hot_encode.fit_transform(weather_data_copy3[['Weather']]) #learns categories and transforms into binary
weather_data_copy3[one_hot_encode.get_feature_names_out(['Weather'])] = encoded #creates now columns with categories and assings encoded value
weather_data_copy3 = weather_data_copy3.drop(columns=['Weather']) #drops the old weather coln that is catgorical from the dataset
weather_data_copy3

,Solar.R,Wind,Month,Day,Year,Temp,Weather_C,Weather_PS,Weather_S
0,190.00,7.4,5,1,2010,67,0.0,0.0,1.0
1,118.00,8.0,5,2,2010,72,1.0,0.0,0.0
2,149.00,12.6,5,3,2010,74,0.0,1.0,0.0
3,313.00,11.5,5,4,2010,62,0.0,0.0,1.0
4,193.33,14.3,5,5,2010,56,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...
153,190.00,7.4,5,1,2010,67,1.0,0.0,0.0
154,193.00,6.9,9,26,2010,70,0.0,1.0,0.0
155,145.00,13.2,9,27,2010,77,0.0,0.0,1.0
156,191.00,14.3,9,28,2010,75,0.0,0.0,1.0
